# 第 1 课：Python 异步、类型与 Pydantic

预计用时：45–60 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 理解 `async`/`await` 解决什么问题
- 用 Pydantic 校验外部输入
- 认识 dataclass、Callable 与类型注解

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 先别急着看代码

这一课只做三件事：

1. 先运行一个普通 Python 函数
2. 再运行一个需要等待的异步函数
3. 最后用 Pydantic 拒绝错误输入

第一次学习时，只要求能按顺序运行并用自己的话解释结果。类、类型注解和异常处理的全部细节，不需要一次记住。


## 本课术语卡

- **函数**：一段可以重复使用的步骤
- **async**：告诉 Python：这个函数可能需要等待
- **await**：等待异步任务完成
- **校验**：使用数据前先检查格式和范围

看到陌生英文时先回到这里。一个术语只需要先记住一句话。


## 推荐学习动作

每个代码格都按这个顺序学习：

1. 先读上方说明，只找“输入”和“输出”。
2. 不修改代码，按 `Shift + Enter` 运行。
3. 看实际结果是否符合说明。
4. 只改一个最小值，再运行一次。

如果报 `NameError`，通常是漏跑了前面的格子；选择 **Restart Kernel and Run All** 可以从头重来。


## 0. 最小热身：先体验普通函数和异步函数

这一段与后面完整工程代码相互独立。先运行它，立刻看到结果。


In [ ]:
import asyncio

def say_hello(name):
    return f"你好，{name}！"

async def wait_and_hello(name):
    await asyncio.sleep(0.1)
    return say_hello(name)

print(say_hello("小明"))
print(await wait_and_hello("小红"))


**你应该观察到什么？**

两个函数得到相似结果。区别是异步函数可以在等待网络或模型时，把运行机会交给其他任务。

如果结果符合说明，再继续下面的完整版本。


## 1. 先理解概念

Agent 经常等待模型和网络接口返回，异步让等待期间可以处理其他任务。Pydantic 把不可信的 JSON 转成经过约束的 Python 对象；这是工具调用的第一道防线。类型注解帮助人和编辑器理解数据契约。

### 本课路线

1. 运行准备单元格
2. 阅读三个核心数据结构
3. 构造合法与非法参数
4. 观察字段范围和额外字段校验
5. 执行一个假的异步工具


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 准备代码


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 核心实验


### 现在做什么？

先给 Agent 装上四个保险丝：最多走几步、模型最多等多久、工具最多等多久、整个任务最多等多久。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class AgentLimits(BaseModel):
    model_config = ConfigDict(extra='forbid')
    max_steps: int = Field(default=8, ge=1, le=30)
    model_timeout_s: float = Field(default=45, gt=0, le=300)
    tool_timeout_s: float = Field(default=15, gt=0, le=120)
    total_timeout_s: float = Field(default=120, gt=0, le=600)


### 现在做什么？

无论工具成功还是失败，都使用同一种返回格式，后面的代码就不必猜测结果长什么样。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class ToolResult(BaseModel):
    ok: bool
    data: Any = None
    error: str | None = None
    retryable: bool = False


### 现在做什么？

先准备变量和依赖

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
ToolHandler = Callable[[BaseModel], Awaitable[Any]]


### 现在做什么？

这像一张工具登记卡：名字、用途、参数格式和真正执行的函数都写在一起。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
@dataclass
class RegisteredTool:
    name: str
    description: str
    args_model: type[BaseModel]
    handler: ToolHandler
    side_effect: bool = False

    def openai_schema(self) -> dict[str, Any]:
        schema = self.args_model.model_json_schema()
        schema['additionalProperties'] = False
        return {
            'type': 'function',
            'function': {
                'name': self.name,
                'description': self.description,
                'parameters': schema,
            },
        }


### 现在做什么？

工具注册表像工具箱目录，负责保存工具、生成说明书并安全执行。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, RegisteredTool] = {}

    def register(self, tool: RegisteredTool) -> None:
        if tool.name in self._tools:
            raise ValueError(f'工具重复注册: {tool.name}')
        self._tools[tool.name] = tool

    @property
    def schemas(self) -> list[dict[str, Any]]:
        return [tool.openai_schema() for tool in self._tools.values()]

    async def execute(self, name: str, raw_arguments: str, timeout_s: float) -> ToolResult:
        tool = self._tools.get(name)
        if tool is None:
            return ToolResult(ok=False, error=f'未知工具: {name}', retryable=False)
        try:
            arguments = json.loads(raw_arguments or '{}')
            validated = tool.args_model.model_validate(arguments)
        except json.JSONDecodeError as exc:
            return ToolResult(ok=False, error=f'工具参数不是合法 JSON: {exc}')
        except ValidationError as exc:
            return ToolResult(ok=False, error=f'工具参数校验失败: {exc}')
        try:
            async with asyncio.timeout(timeout_s):
                value = await tool.handler(validated)
            return ToolResult(ok=True, data=value)
        except TimeoutError:
            return ToolResult(ok=False, error=f'工具 {name} 执行超时', retryable=True)
        except httpx.HTTPStatusError as exc:
            retryable = exc.response.status_code in {408, 429, 500, 502, 503, 504}
            return ToolResult(ok=False, error=f'上游 HTTP {exc.response.status_code}', retryable=retryable)
        except Exception as exc:
            return ToolResult(ok=False, error=f'{type(exc).__name__}: {exc}', retryable=False)


## 3. 动手验证

运行下面的小实验，先猜测输出，再执行。


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
limits = AgentLimits(max_steps=3, tool_timeout_s=2)
print(limits.model_dump())

try:
    AgentLimits(max_steps=0)
except ValidationError as exc:
    print('预期中的校验错误：', exc.errors()[0]['msg'])


## 4. 代码讲解

`AgentLimits` 集中管理循环、模型和工具超时；`ToolResult` 统一成功与失败形状；`RegisteredTool` 把名称、说明、参数模型和执行函数组合成一个可注册工具。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 给 `AgentLimits` 增加 `max_retries` 字段并限制为 0–5
- 创建一个只接受正整数的 Pydantic 模型
- 用 `asyncio.sleep` 模拟一个耗时 0.1 秒的工具

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 能说出协程与普通函数的调用差异
- [ ] 非法参数会在业务函数执行前被拒绝
- [ ] 理解 `extra='forbid'` 的意义

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `02_最小Agent循环.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。


## 一句话回顾

异步负责等待，Pydantic 负责把关；两者都是 Agent 调用外部服务时的基础。

### 如果你仍然觉得难

先只完成以下最低目标：

- 能从上到下运行本课；
- 能指出哪一格是输入、哪一格产生输出；
- 能用一句话说出本课解决了什么问题。

做到这三点就可以进入下一课。第二遍学习时再研究类型注解、异常分支和工程细节。
